# Rugo — Worked Examples

[**rugo**](https://pypi.org/project/rugo/) is a columnar file engine for Parquet, CSV, and JSONL —
no PyArrow, no NumPy, no heavy dependencies. Below are worked examples of the
most common things people do with it: reading, selecting columns, filtering,
summarizing, sorting, and writing data.

Dataset: [Space Missions](https://storage.googleapis.com/opteryx/rugo_examples/space_missions.parquet) —
every orbital launch attempt from 1957 to 2022, one row per launch.


In [1]:
!pip install --upgrade rugo

In [2]:
import os

DATA_URL = "https://storage.googleapis.com/opteryx/rugo_examples/space_missions.parquet"
DATA_FILE = "space_missions.parquet"  # every cell below reads from this path

if not os.path.exists(DATA_FILE):
    !wget -q {DATA_URL}  # skip the download if it's already sitting next to the notebook

print("Dataset ready:", os.path.getsize(DATA_FILE), "bytes")

Dataset ready: 92836 bytes


## 1. Schema — inspect a file without reading any data

This is an example of a footer-only read: `read_metadata` parses just the
Parquet footer, so it's fast and cheap even on huge files — no column data
is decoded. Each column has both a physical type (how it's stored on disk)
and a logical type (what it actually represents) — `Launched_at` is a good
example: it's stored as `int64` but its logical type shows it's really a
`timestamp[us]`.


In [3]:
from rugo import parquet

meta = parquet.read_metadata(DATA_FILE)  # footer only — no column data is touched

print(f"Rows : {meta.num_rows:,}")
print(f"Columns ({len(meta.schema_columns)}):")
for col in meta.schema_columns:
    nullable = "nullable" if col.nullable else "not null"
    # physical_type is how the column is stored on disk; logical_type is what it means
    print(f"  {col.name:<30}  {col.physical_type:<12} {col.logical_type:<14} ({nullable})")

Rows : 4,630
Columns (8):
  Company                         byte_array   varchar        (nullable)
  Location                        byte_array   varchar        (nullable)
  Price                           float64      float64        (nullable)
  Launched_at                     int64        timestamp[us]  (nullable)
  Rocket                          byte_array   varchar        (nullable)
  Rocket_Status                   byte_array   varchar        (nullable)
  Mission                         byte_array   varchar        (nullable)
  Mission_Status                  byte_array   varchar        (nullable)


## 2. Read — pull the data in as Morsels

This is an example of streaming a file into **Morsels** — one per row group —
instead of loading a single in-memory table. Memory stays flat no matter how
large the file is.


In [4]:
from rugo import parquet

with parquet.read_parquet(DATA_FILE) as reader:  # nothing is decoded yet — this just opens the file
    for morsel in reader:  # each iteration decodes and returns the next row group
        print(f"morsel: {morsel.num_rows:,} rows x {morsel.num_columns} columns")

# this file happens to fit in one row group/morsel — larger files stream as
# several, processed one at a time without ever holding the whole table in
# memory.

morsel: 4,630 rows x 8 columns


## 3. Select columns — read only what you need

This is an example of column selection: pass `columns=` and everything else
is skipped during decode. Fewer columns read means less I/O and less CPU.


In [5]:
from rugo import parquet

with parquet.read_parquet(DATA_FILE, columns=["Company", "Mission", "Price"]) as reader:
    morsel = next(iter(reader))  # one row group is enough to look at

for name in morsel.column_names:  # only the 3 requested columns are present, in that order
    vec = morsel.column(name)  # a typed column Vector, not a plain Python list
    label = name.decode() if isinstance(name, bytes) else name  # column names come back as bytes
    print(f"{label:<10}  type={vec.type.name:<10}  first 3: {vec.to_pylist()[:3]}")

Company     type=VARCHAR     first 3: ['RVSN USSR', 'RVSN USSR', 'US Navy']
Mission     type=VARCHAR     first 3: ['Sputnik-1', 'Sputnik-2', 'Vanguard TV3']
Price       type=FLOAT64     first 3: [None, None, None]


## 4. Filter — narrow the data as it's read

This is an example of filtering during the read itself, via `predicates=`
(rugo's term for a filter condition). rugo first uses row-group statistics to
skip whole row groups that can't match, then filters the remaining rows as
they're decoded — rows that don't match are never fully materialized.


In [6]:
from rugo import parquet

# "predicates" is rugo's parameter name for a list of filter conditions
with parquet.read_parquet(DATA_FILE, predicates=[("Company", "==", "SpaceX")]) as reader:
    morsel = next(iter(reader))

print(morsel)  # only SpaceX rows made it through

companies = morsel.column("Company")
print(companies)  # a typed Vector, not a plain Python list
print(companies.to_pylist()[:10])  # convert to a plain list when you need one

┌────┬─────────┬────────────────────────────────┬─────────┬────────────────────────────┬────────────────────┬───────────────┬────────────────────────────────┬───────────────────┐
│    │ Company │            Location            │  Price  │        Launched_at         │       Rocket       │ Rocket_Status │            Mission             │   Mission_Status  │
│    │ VARCHAR │            VARCHAR             │ FLOAT64 │        TIMESTAMP64         │      VARCHAR       │    VARCHAR    │            VARCHAR             │      VARCHAR      │
╞════╪═════════╪════════════════════════════════╪═════════╪════════════════════════════╪════════════════════╪═══════════════╪════════════════════════════════╪═══════════════════╡
│  1 │ SpaceX  │ Omelek Island, Ronald Reagan … │ 7.0     │ 2006-03-24 21:30:00+00:00  │ Falcon 1           │ Retired       │ FalconSat-2                    │ Failure           │
│  2 │ SpaceX  │ Omelek Island, Ronald Reagan … │ 7.0     │ 2007-03-21 01:10:00+00:00  │ Falcon 1        

## 5. Iterate rows — as named tuples

This is an example of row-at-a-time access: a Morsel iterates as named tuples
with typed attributes, so no manual column zipping is required.


In [7]:
from rugo import parquet

with parquet.read_parquet(
    DATA_FILE,
    columns=["Company", "Location", "Price"],
    predicates=[("Company", "==", "Sandia")],
) as reader:
    for morsel in reader:
        for row in morsel:  # each row comes out as a named tuple
            print(row)
            print(row.Company, "@", row.Location)  # typed attribute access, no indexing needed

Row(Company='Sandia', Location='LP-41, Kauai, Pacific Missile Range Facility', Price=15.0)
Sandia @ LP-41, Kauai, Pacific Missile Range Facility


## 6. Summarize — total launch spend by company

This is an example of a running summary: column selection and filtering are
handled by rugo, and a plain Python `dict` accumulates totals one morsel at a
time, so the full table is never held in memory.


In [8]:
from collections import defaultdict
from rugo import parquet

def _str(v):
    return v.decode() if isinstance(v, (bytes, bytearray)) else v  # column values come back as bytes

totals = defaultdict(float)
counts = defaultdict(int)

with parquet.read_parquet(DATA_FILE, columns=["Company", "Price"]) as reader:
    for morsel in reader:
        # zip pairs each row's Company with its Price, column by column
        for company, price in zip(morsel.column("Company"), morsel.column("Price")):
            company = _str(company)
            counts[company] += 1
            if price is not None:  # Price is nullable — some missions never disclosed a cost
                totals[company] += price

ranked = sorted(totals.items(), key=lambda kv: kv[1], reverse=True)[:10]  # top 10 by total spend
print(f"{'Company':<35}  {'Missions':>8}  {'Total Price ($M)':>16}")
print("-" * 64)
for company, total in ranked:
    print(f"{company:<35}  {counts[company]:>8,}  {total:>16,.1f}")

Company                              Missions  Total Price ($M)
----------------------------------------------------------------
NASA                                      203          61,200.0
Arianespace                               293          18,173.0
ULA                                       151          16,552.0
SpaceX                                    182          11,181.0
CASC                                      338           9,352.2
Northrop                                   89           4,350.0
MHI                                        87           3,712.5
ISRO                                       82           2,307.0
VKS RF                                    216           1,901.7
US Air Force                              161           1,550.9


## 7. Sort — compute an order, then apply it

This is an example of sorting in two steps. `morsel_sort` figures out the
new row order without touching the Morsel itself — it returns a list where
position `i` holds the index of the row that belongs there. `.take()` then
applies that order to produce the sorted rows.


In [9]:
from draken.morsels.sort import morsel_sort
from rugo import parquet

with parquet.read_parquet(DATA_FILE, columns=["Company", "Mission"]) as reader:
    morsel = next(iter(reader))

print("original order:", morsel.column("Company").to_pylist()[:5])  # on-disk order

perm = morsel_sort(morsel, column_names=["Company"], ascending=[True])  # order only — no rows moved yet
sorted_morsel = morsel.take(perm)  # apply the order to produce the sorted rows

print("sorted by Company:", sorted_morsel.column("Company").to_pylist()[:5])

original order: ['RVSN USSR', 'RVSN USSR', 'US Navy', 'AMBA', 'US Navy']
sorted by Company: ['AEB', 'AEB', 'AEB', 'AMBA', 'AMBA']


## 8. Write — straight from a Morsel

This is an example of writing a Morsel out directly: `write_jsonl` serializes
it to bytes in C++, with no Python `json` module involved (`write_csv` does
the same for CSV). Here every row group of the filtered result streams
straight to a JSONL file.


In [10]:
import os
from rugo import parquet
from rugo.jsonl import write_jsonl

with open("spacex_missions.jsonl", "wb") as f:  # binary mode — write_jsonl returns bytes
    with parquet.read_parquet(DATA_FILE, predicates=[("Company", "==", "SpaceX")]) as reader:
        for morsel in reader:
            f.write(write_jsonl(morsel))  # one row group in, one JSONL chunk out

print(f"Written {os.path.getsize('spacex_missions.jsonl'):,} bytes to spacex_missions.jsonl")

Written 43,306 bytes to spacex_missions.jsonl


In [11]:
!head -n 3 spacex_missions.jsonl

{"Company":"SpaceX","Location":"Omelek Island, Ronald Reagan Ballistic Missile Defense Test Site, Marshall Islands, USA","Price":7.0,"Launched_at":"2006-03-24T21:30:00+00:00","Rocket":"Falcon 1","Rocket_Status":"Retired","Mission":"FalconSat-2","Mission_Status":"Failure"}
{"Company":"SpaceX","Location":"Omelek Island, Ronald Reagan Ballistic Missile Defense Test Site, Marshall Islands, USA","Price":7.0,"Launched_at":"2007-03-21T01:10:00+00:00","Rocket":"Falcon 1","Rocket_Status":"Retired","Mission":"DemoSat","Mission_Status":"Failure"}
{"Company":"SpaceX","Location":"Omelek Island, Ronald Reagan Ballistic Missile Defense Test Site, Marshall Islands, USA","Price":7.0,"Launched_at":"2008-08-03T03:34:00+00:00","Rocket":"Falcon 1","Rocket_Status":"Retired","Mission":"Flight 3","Mission_Status":"Failure"}


## 9. Interop — convert to a PyArrow Table

This is an example of handing data off to an existing PyArrow-based
toolchain: a Morsel converts to an Arrow Table on request, via the C Data
Interface, zero-copy where the layouts allow it. This is the only place in
this notebook that touches PyArrow — everything above ran without it.


In [12]:
import pyarrow
from rugo import parquet

with parquet.read_parquet(DATA_FILE, columns=["Company", "Mission", "Price"]) as reader:
    morsel = next(iter(reader))  # a rugo Morsel — still no PyArrow involved yet

print(morsel.to_arrow())  # zero-copy handoff via the Arrow C Data Interface

pyarrow.Table
Company: string
Mission: string
Price: double
----
Company: [["RVSN USSR","RVSN USSR","US Navy","AMBA","US Navy",...,"SpaceX","CASC","SpaceX","CAS Space","CASC"]]
Mission: [["Sputnik-1","Sputnik-2","Vanguard TV3","Explorer 1","Vanguard TV3BU",...,"Starlink Group 3-2","Wentian","Starlink Group 4-25","Demo Flight","Yaogan 35 Group 03"]]
Price: [[null,null,null,null,null,...,67,null,67,null,29.75]]


## Next steps

More focused, runnable examples — CSV/JSONL predicate pushdown, constant-memory
streaming writes, the raw Morsel API — live in
[`rugo/examples/`](https://github.com/mabel-dev/opteryx-core/tree/main/rugo/examples).
